# Visualising objects

In [ ]:

# =========================
# Paths & Parameters (Edit this)
# =========================
from pathlib import Path
ROOT_ANALYSIS = Path(r"path/to/dataset_folder")
ROOT_ANN      = ROOT_ANALYSIS / "annotations_gbr"
ROOT_SPLIT    = ROOT_ANALYSIS / "split_nuclei_membrane_raw"
ROOT_MEM_ROI  = ROOT_ANALYSIS / "membrane_roi"
ROOT_SEG_MEM  = ROOT_ANALYSIS / "seg_membrane_timelapses_fixed"
ROOT_SEG_NUC  = ROOT_ANALYSIS / "seg_nuclei_timelapses"
ROOT_ANIM     = ROOT_ANALYSIS / "animation_gbr"
ROOT_ANIM_NUC = ROOT_ANALYSIS / "animation_gbr_nuc"
ROOT_TRACKS   = ROOT_ANALYSIS / "gbr_tracks_animation"
PATHS = {
    "full_df": ROOT_ANALYSIS / "nuclei_membrane_tracking" / "full_manual_dataset_cropped.csv",
    "gt_t285": ROOT_ANN / "gbr_annotations_t285.csv",
    "t285_nuc": ROOT_SPLIT / "VollSeg" / "StarDist" / "Merged-285.tif",
    "sectors_t359_1": ROOT_MEM_ROI / "sectors" / "Merged_359_1.tif",
    "sectors_t359_2": ROOT_MEM_ROI / "sectors" / "Merged_359_2.tif",
    "sectors_t359_3": ROOT_MEM_ROI / "sectors" / "Merged_359_3.tif",
    "timelapse_mem_seg": ROOT_SEG_MEM / "timelapse_sixth_dataset.tif",
    "timelapse_nuc_seg": ROOT_SEG_NUC / "timelapse_sixth_dataset.tif",
    "selected_tracks_csv": ROOT_ANALYSIS / "selected_tracks_df.csv",
    "croppings_mem_tif":   ROOT_ANALYSIS / "croppings_array_mem.tif",
    "croppings_nuc_tif":   ROOT_ANALYSIS / "croppings_array_nuc.tif",
    "anim_radial":     ROOT_ANIM / "radial",
    "anim_basal":      ROOT_ANIM / "basal",
    "anim_goblet":     ROOT_ANIM / "goblet",
    "anim_radial_pc":  ROOT_ANIM / "radial_pointcloud",
    "anim_basal_pc":   ROOT_ANIM / "basal_pointcloud",
    "anim_goblet_pc":  ROOT_ANIM / "goblet_pointcloud",
    "anim_nuc_radial":    ROOT_ANIM_NUC / "radial",
    "anim_nuc_basal":     ROOT_ANIM_NUC / "basal",
    "anim_nuc_goblet":    ROOT_ANIM_NUC / "goblet",
    "anim_nuc_radial_pc": ROOT_ANIM_NUC / "radial_pointclouds",
    "anim_nuc_basal_pc":  ROOT_ANIM_NUC / "basal_pointclouds",
    "anim_nuc_goblet_pc": ROOT_ANIM_NUC / "goblet_pointclouds",
    "tracks_anim_dir": ROOT_TRACKS,
}
for key in ["anim_radial","anim_basal","anim_goblet","anim_radial_pc","anim_basal_pc","anim_goblet_pc","anim_nuc_radial","anim_nuc_basal","anim_nuc_goblet","anim_nuc_radial_pc","anim_nuc_basal_pc","anim_nuc_goblet_pc","tracks_anim_dir"]:
    PATHS[key].mkdir(parents=True, exist_ok=True)
N_TIMEPOINTS = 360
CROP_Z_MEM   = 10
CROP_Z_NUC   = 20
CROP_XY      = 150


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
from tqdm import tqdm

## Load inputs

In [ ]:
full_df = pd.read_csv(str(PATHS["full_df"]))

In [ ]:
import tifffile as tiff
t285_nuc_array = tiff.imread(str(PATHS["t285_nuc"]))

In [ ]:
gt_df = pd.read_csv(str(PATHS["gt_t285"]))

## Label lookup @ t=285 (nucleus labels at annotated points)

In [ ]:
# Use the rounded coordinates to extract values from the 3D array
gt_df['Label'] = gt_df.apply(
    lambda row: t285_nuc_array[int(row['axis-0']), int(row['axis-1']), int(row['axis-2'])], axis=1
)

In [ ]:
full_df_t285 = full_df[full_df['Spot frame'] == 285]

In [ ]:
# Step 2: Merge the DataFrames based on the rounded columns
concatenated_merged_final = pd.merge(full_df_t285, gt_df[['Label', 'gbr_cell_type']], left_on='nuc_label', right_on='Label', how='left')

In [ ]:
def determine_cell_type(group):
    cell_types = set(group['gbr_cell_type'])

    # Hierarchical rules    
    # Rule 1: If all rows with the same Spot Track ID have the same cell_type value, use that value
    if len(cell_types) == 1:
        # Check for NaN in the cell_type
        single_cell_type = list(cell_types)[0]
        if pd.notna(single_cell_type):  # Use pd.notna() to check for non-NaN values
            return single_cell_type
        else:
            return 'unknown'
    
    # Rule 2: If any rows have 'mcc', 'ic', or 'ssc', use that value:
    if 'radial' in cell_types:
            return 'radial'

    # Rule 3: If both 'basal' and 'goblet' are present, use 'unknown'
    if 'basal' in cell_types and 'goblet' in cell_types:
        return 'unknown'
    
    # Rule 4: If either 'basal' or 'goblet' is present, use that value
    if 'basal' in cell_types:
        return 'basal'
    if 'goblet' in cell_types:
        return 'goblet'
    
    # Rule 5: If none of the above, use 'unknown'
    return 'unknown'

# Group by 'Spot Track ID' and apply the hierarchical function
mapping_dict = concatenated_merged_final.groupby('Spot track ID relabelled').apply(determine_cell_type).to_dict()

In [ ]:
# Create the 'cell_type' column by mapping from 'Spot track ID relabelled'
full_df['gbr_cell_type'] = full_df['Spot track ID relabelled'].map(mapping_dict)


In [ ]:
full_df2 = full_df[(full_df['gbr_cell_type'] != 'unknown') & ~(pd.isna(full_df['gbr_cell_type']))]

## Filter & select cells for t=359

In [ ]:
full_df2_final = full_df2[full_df2['t'] == 359]

In [ ]:
sector1 = tiff.imread(str(PATHS["sectors_t359_1"]))
sector2 = tiff.imread(str(PATHS["sectors_t359_2"]))
sector3 = tiff.imread(str(PATHS["sectors_t359_3"]))

## Build lineages & select top tracks per cell type

In [ ]:
radial_sector_df = full_df2_final[full_df2_final['gbr_cell_type'] == 'radial']
# Convert coordinates to integers (indices)
x_int = radial_sector_df['X_orig'].astype(int)
y_int = radial_sector_df['Y_orig'].astype(int)

# Ensure coordinates are within image bounds
valid = (x_int >= 0) & (x_int < sector1.shape[1]) & (y_int >= 0) & (y_int < sector1.shape[0])

# Subset to only valid coordinates
x_valid = x_int[valid]
y_valid = y_int[valid]
df_valid = radial_sector_df[valid]

# Check mask value at each coordinate
inside_mask = sector1[y_valid, x_valid] > 0  # or == 1

# Final filtered DataFrame
radial_sector_df_inside = df_valid[inside_mask]
radial_track_ids = radial_sector_df_inside['Spot track ID relabelled'].unique()
full_df2[full_df2['Spot track ID relabelled'].isin(radial_track_ids)].groupby(['Spot track ID relabelled'])['t_hours'].nunique().reset_index(name='unique_t_hours')

In [ ]:
goblet_sector_df = full_df2_final[full_df2_final['gbr_cell_type'] == 'goblet']
# Convert coordinates to integers (indices)
x_int = goblet_sector_df['X_orig'].astype(int)
y_int = goblet_sector_df['Y_orig'].astype(int)

# Ensure coordinates are within image bounds
valid = (x_int >= 0) & (x_int < sector2.shape[1]) & (y_int >= 0) & (y_int < sector2.shape[0])

# Subset to only valid coordinates
x_valid = x_int[valid]
y_valid = y_int[valid]
df_valid = goblet_sector_df[valid]

# Check mask value at each coordinate
inside_mask = sector2[y_valid, x_valid] > 0  # or == 1

# Final filtered DataFrame
goblet_sector_df_inside = df_valid[inside_mask]
goblet_track_ids = goblet_sector_df_inside['Spot track ID relabelled'].unique()
full_df2[full_df2['Spot track ID relabelled'].isin(goblet_track_ids)].groupby(['Spot track ID relabelled'])['t_hours'].nunique().reset_index(name='unique_t_hours')

In [ ]:
basal_sector_df = full_df2_final[full_df2_final['gbr_cell_type'] == 'basal']
# Convert coordinates to integers (indices)
x_int = basal_sector_df['X_orig'].astype(int)
y_int = basal_sector_df['Y_orig'].astype(int)

# Ensure coordinates are within image bounds
valid = (x_int >= 0) & (x_int < sector3.shape[1]) & (y_int >= 0) & (y_int < sector3.shape[0])

# Subset to only valid coordinates
x_valid = x_int[valid]
y_valid = y_int[valid]
df_valid = basal_sector_df[valid]

# Check mask value at each coordinate
inside_mask = sector3[y_valid, x_valid] > 0  # or == 1

# Final filtered DataFrame
basal_sector_df_inside = df_valid[inside_mask]
basal_track_ids = basal_sector_df_inside['Spot track ID relabelled'].unique()
full_df2[full_df2['Spot track ID relabelled'].isin(basal_track_ids)].groupby(['Spot track ID relabelled'])['t_hours'].nunique().reset_index(name='unique_t_hours')

In [ ]:
selected_df = full_df2[full_df2['Spot track ID relabelled'].isin([63, 2690, 225])]

## Prepare per-track subsets for membrane & nucleus

In [ ]:
def construct_lineages(df, complete_df):
    # Extract the last timepoint dataframe
    last_timepoint_df = df[df['t'] == df['t'].max()]
    last_ids = last_timepoint_df[['ID', 'gbr_cell_type', 'Spot track ID relabelled']]

    # Initialize the lineages dictionary
    lineages = {}

    # Iterate over each ID in the last timepoint
    for _, row in last_ids.iterrows():
        current_id = row['ID']
        cell_type = row['gbr_cell_type']
        track_id = row['Spot track ID relabelled']
        lineage_ids = []
        n_mem_labels = 0

        print(f'Processing id {current_id}')

        # Trace lineage
        while not pd.isna(current_id):
            lineage_ids.append(current_id)
            #print(current_id)

            if len(complete_df[complete_df['ID'] == current_id]) > 0:
                # Find the row corresponding to the current ID
                current_row = complete_df[complete_df['ID'] == current_id].iloc[0]

                # Check mem_label
                if not pd.isna(current_row['mem_label']) and current_row['mem_label'] != 0:
                    n_mem_labels += 1

                # Move to the source_ID
                current_id = current_row['Spot source ID']
            else:
                current_id = pd.NA

        # Add to lineages dictionary
        lineages[row['ID']] = {
            'gbr_cell_type': cell_type,
            'lineage_ids': lineage_ids,
            'track_id' : track_id,
            'n_mem_labels': n_mem_labels
        }

    return lineages

lineages = construct_lineages(selected_df, full_df)



In [ ]:
def construct_nucleus_lineages(df, complete_df):
    # Extract the last timepoint dataframe
    last_timepoint_df = df[df['t'] == df['t'].max()]
    last_ids = last_timepoint_df[['ID', 'gbr_cell_type', 'Spot track ID relabelled']]

    # Initialize the lineages dictionary
    lineages = {}

    # Iterate over each ID in the last timepoint
    for _, row in last_ids.iterrows():
        current_id = row['ID']
        cell_type = row['gbr_cell_type']
        track_id = row['Spot track ID relabelled']
        lineage_ids = []
        n_nuc_labels = 0

        print(f'Processing id {current_id}')

        # Trace lineage
        while not pd.isna(current_id):
            lineage_ids.append(current_id)
            #print(current_id)

            if len(complete_df[complete_df['ID'] == current_id]) > 0:
                # Find the row corresponding to the current ID
                current_row = complete_df[complete_df['ID'] == current_id].iloc[0]

                # Check nuc_label
                if not pd.isna(current_row['nuc_label']) and current_row['nuc_label'] != 0:
                    n_nuc_labels += 1

                # Move to the source_ID
                current_id = current_row['Spot source ID']
            else:
                current_id = pd.NA

        # Add to lineages dictionary
        lineages[row['ID']] = {
            'gbr_cell_type': cell_type,
            'lineage_ids': lineage_ids,
            'track_id' : track_id,
            'n_nuc_labels': n_nuc_labels
        }

    return lineages

nuc_lineages = construct_nucleus_lineages(selected_df, full_df)



In [ ]:
def find_top_lineages_and_subset_df(df, lineages, top_n=10):
    # Convert lineages dictionary to a dataframe
    lineages_df = pd.DataFrame([
        {'ID': key, 'gbr_cell_type': value['gbr_cell_type'], 'n_mem_labels': value['n_mem_labels'], 'lineage_ids': value['lineage_ids']}
        for key, value in lineages.items()
    ])

    # Find top IDs by n_mem_labels for each cell_type
    top_lineages = (
        lineages_df.sort_values(['gbr_cell_type', 'n_mem_labels'], ascending=[True, False])
        .groupby('gbr_cell_type')
        .head(top_n)
    )

    # Initialize an empty list to store rows for the subset dataframe
    rows = []

    # Extract relevant rows for each selected lineage
    for _, row in top_lineages.iterrows():
        lineage_ids = row['lineage_ids']
        # Get rows from the original dataframe corresponding to lineage IDs
        lineage_rows = df[df['ID'].isin(lineage_ids)][['Spot track ID relabelled', 'Track ID_y', 'gbr_cell_type', 't', 'mem_label', 'mem_x', 'mem_y', 'mem_z']]
        rows.append(lineage_rows)

    # Combine all rows into a subset dataframe
    subset_df = pd.concat(rows, ignore_index=True)

    return subset_df


In [ ]:
lineages_df = pd.DataFrame([
        {
            'ID': key,
            'gbr_cell_type': value['gbr_cell_type'],
            'n_mem_labels': value['n_mem_labels'],
            'lineage_ids': value['lineage_ids'],
            'track_id': value['track_id']
        }
        for key, value in lineages.items()])

lineages_df

In [ ]:
# For each track_id, keep the row with the highest n_mem_labels
lineages_df = (
    lineages_df.sort_values('n_mem_labels', ascending=False)
    .drop_duplicates(subset=['track_id'], keep='first')
)

# Find the top `top_n` IDs for each cell_type by n_mem_labels
top_lineages = (
    lineages_df.sort_values(['gbr_cell_type', 'n_mem_labels'], ascending=[True, False])
    .groupby('gbr_cell_type')
    .head(3)
)

top_lineages

In [ ]:
rows = []
# Extract relevant rows for each selected lineage
for _, row in top_lineages.iterrows():
    lineage_ids = row['lineage_ids']
    # Get rows from the original dataframe corresponding to lineage IDs
    lineage_rows = full_df2[full_df2['ID'].isin(lineage_ids)][['Spot track ID relabelled', 'Track ID_y', 'gbr_cell_type', 't', 'mem_label', 'mem_x', 'mem_y', 'mem_z']]
    rows.append(lineage_rows)

# Combine all rows into a subset dataframe
subset_df = pd.concat(rows, ignore_index=True)

subset_df

In [ ]:
rows = []
# Extract relevant rows for each selected lineage
for _, row in top_lineages.iterrows():
    lineage_ids = row['lineage_ids']
    # Get rows from the original dataframe corresponding to lineage IDs
    lineage_rows = full_df2[full_df2['ID'].isin(lineage_ids)][['Spot track ID relabelled', 'Track ID_y', 'gbr_cell_type', 't', 'nuc_label', 'X_orig', 'Y_orig', 'Z_orig']]
    rows.append(lineage_rows)

# Combine all rows into a subset dataframe
subset_df_nuc = pd.concat(rows, ignore_index=True)

subset_df_nuc.to_csv(str(PATHS["selected_tracks_csv"]))

In [ ]:
# Step 1: Get sorted unique values
unique_sorted = sorted(subset_df['Spot track ID relabelled'].unique())

# Step 2: Create mapping dictionary
value_map = {value: idx for idx, value in enumerate(unique_sorted)}

# Step 3: Apply the mapping
subset_df['Track ID_remapped'] = subset_df['Spot track ID relabelled'].map(value_map)

In [ ]:
# Step 1: Get sorted unique values
unique_sorted = sorted(subset_df_nuc['Spot track ID relabelled'].unique())

# Step 2: Create mapping dictionary
value_map = {value: idx for idx, value in enumerate(unique_sorted)}

# Step 3: Apply the mapping
subset_df_nuc['Track ID_remapped'] = subset_df_nuc['Spot track ID relabelled'].map(value_map)

## 4D croppings around each cell (membrane)

In [ ]:
full_mem_array = tiff.imread(str(PATHS["timelapse_mem_seg"]))

In [ ]:
# Pad the z-axis of the original image array with 3 slices in both directions
padded_mem_array = np.pad(
    full_mem_array,
    pad_width=((0, 0), (3, 3), (0, 0), (0, 0)),  # Only pad the Z-axis
    mode='constant',  # Use constant values (e.g., zero padding)
    constant_values=0  # Pad with zeros
)

In [ ]:
del full_mem_array

In [ ]:
import gc
gc.collect()

In [ ]:
def generate_croppings(df, image_array):
    N, T, Z, Y, X = 3, 360, 10, 150, 150
    croppings = np.zeros((N, T, Z, Y, X), dtype=bool)

    # Process each Track ID group
    for n_idx, (track_id, group) in enumerate(df.groupby('Track ID_remapped')):
        print(f'Processing Track ID: {track_id}')

        for t in range(T):  # Iterate over timepoints
            if t in group['t'].values:
                row = group[group['t'] == t].iloc[0]
                
                mem_label, mem_x, mem_y, mem_z = row['mem_label'], row['mem_x'], row['mem_y'], row['mem_z']

                if not pd.isna(mem_x):
                    mem_label = int(mem_label)
                    mem_x = int(mem_x)
                    mem_y = int(mem_y)
                    mem_z = int(mem_z)

                    # Define cropping bounds
                    z_min, z_max = mem_z - 5, mem_z + 5
                    y_min, y_max = mem_y - 75, mem_y + 75
                    x_min, x_max = mem_x - 75, mem_x + 75

                    # Ensure bounds stay within image dimensions
                    z_min_pad, z_max_pad = max(0, z_min), min(image_array.shape[1], z_max)
                    y_min_pad, y_max_pad = max(0, y_min), min(image_array.shape[2], y_max)
                    x_min_pad, x_max_pad = max(0, x_min), min(image_array.shape[3], x_max)

                    # Extract crop
                    crop = image_array[
                        t, z_min_pad:z_max_pad, y_min_pad:y_max_pad, x_min_pad:x_max_pad
                    ] == mem_label

                    # Pad crop to (10, 150, 150)
                    crop = np.pad(
                        crop,
                        pad_width=(
                            (0, Z - crop.shape[0]),
                            (0, Y - crop.shape[1]),
                            (0, X - crop.shape[2])
                        ),
                        mode='constant',
                        constant_values=0
                    )

                    # Assign to croppings array
                    croppings[n_idx, t] = crop
                else:
                    continue
            else:
                continue

    return croppings


In [ ]:
croppings_array = generate_croppings(subset_df, padded_mem_array)

In [ ]:
#save croppings array as .tif
tiff.imwrite(str(PATHS["croppings_mem_tif"]), croppings_array.astype(np.uint8))

## 4D croppings around each cell (nucleus)

In [ ]:
del padded_mem_array

gc.collect()

In [ ]:
def generate_croppings_nuc(df, image_array):
    N, T, Z, Y, X = 3, 360, 20, 150, 150
    croppings = np.zeros((N, T, Z, Y, X), dtype=bool)

    # Process each Track ID group
    for n_idx, (track_id, group) in enumerate(df.groupby('Track ID_remapped')):
        print(f'Processing Track ID: {track_id}')

        for t in range(T):  # Iterate over timepoints
            if t in group['t'].values:
                row = group[group['t'] == t].iloc[0]
                
                nuc_label, nuc_x, nuc_y, nuc_z = row['nuc_label'], row['X_orig'], row['Y_orig'], row['Z_orig']

                if not pd.isna(nuc_x):
                    nuc_label = int(nuc_label)
                    nuc_x = int(nuc_x)
                    nuc_y = int(nuc_y)
                    nuc_z = int(nuc_z)

                    # Define cropping bounds
                    z_min, z_max = nuc_z - 10, nuc_z + 10
                    y_min, y_max = nuc_y - 75, nuc_y + 75
                    x_min, x_max = nuc_x - 75, nuc_x + 75

                    # Ensure bounds stay within image dimensions
                    z_min_pad, z_max_pad = max(0, z_min), min(image_array.shape[1], z_max)
                    y_min_pad, y_max_pad = max(0, y_min), min(image_array.shape[2], y_max)
                    x_min_pad, x_max_pad = max(0, x_min), min(image_array.shape[3], x_max)

                    # Extract crop
                    crop = image_array[
                        t, z_min_pad:z_max_pad, y_min_pad:y_max_pad, x_min_pad:x_max_pad
                    ] == nuc_label

                    # Pad crop to (10, 150, 150)
                    crop = np.pad(
                        crop,
                        pad_width=(
                            (0, Z - crop.shape[0]),
                            (0, Y - crop.shape[1]),
                            (0, X - crop.shape[2])
                        ),
                        mode='constant',
                        constant_values=0
                    )

                    # Assign to croppings array
                    croppings[n_idx, t] = crop
                else:
                    continue
            else:
                continue

    return croppings


In [ ]:
full_nuc_array = tiff.imread(str(PATHS["timelapse_nuc_seg"]))

In [ ]:
# Pad the z-axis of the original image array with 3 slices in both directions
padded_nuc_array = np.pad(
    full_nuc_array,
    pad_width=((0, 0), (3, 3), (0, 0), (0, 0)),  # Only pad the Z-axis
    mode='constant',  # Use constant values (e.g., zero padding)
    constant_values=0  # Pad with zeros
)

In [ ]:
del full_nuc_array

gc.collect()

In [ ]:
croppings_array_nuc = generate_croppings_nuc(subset_df_nuc, padded_nuc_array)

In [ ]:
#save intermediate croppings array to avoid having to regenerate
tiff.imwrite(str(PATHS["croppings_nuc_tif"]), croppings_array_nuc.astype(np.uint8))

In [ ]:
del padded_nuc_array

gc.collect()

## 3D point clouds (marching cubes + trimesh sampling)

## Napari views & animation export (membrane)

In [ ]:
celltype_dict = {0 : 'radial', 1 : 'basal', 2 : 'goblet'}

In [ ]:
%%capture
#import ncolor
mem_mask = croppings_array
#mask_nc = ncolor.label(mask,max_depth=20)

import napari
viewer = napari.view_labels(mem_mask)
voxel_size_x = 0.691 # um
voxel_size_y = 0.691 # um
voxel_size_z = 2 # um

reference_size = voxel_size_x

factor_z = voxel_size_z / reference_size
factor_y = voxel_size_y / reference_size
factor_x = voxel_size_x / reference_size

#viewer.layers['mem_mask'].scale = [factor_z, factor_y, factor_x] # Z, Y, X order
viewer.dims.ndisplay = 3
#viewer.camera.center = [s//2 for s in mask.shape]
viewer.camera.zoom=5
viewer.camera.angles=(10.90517458968619, -20.777067798396835, 58.04311170773853)
viewer.camera.perspective=0.0
#viewer.camera.interactive=True


In [ ]:
%%capture
#import ncolor
mem_mask = croppings_array
#mask_nc = ncolor.label(mask,max_depth=20)

import napari
viewer = napari.view_labels(mem_mask)
voxel_size_x = 0.691 # um
voxel_size_y = 0.691 # um
voxel_size_z = 2 # um

reference_size = voxel_size_x

factor_z = voxel_size_z / reference_size
factor_y = voxel_size_y / reference_size
factor_x = voxel_size_x / reference_size

#viewer.layers['mem_mask'].scale = [factor_z, factor_y, factor_x] # Z, Y, X order
viewer.dims.ndisplay = 3
#viewer.camera.center = [s//2 for s in mask.shape]
viewer.camera.zoom=5
viewer.camera.angles=(10.90517458968619, -20.777067798396835, 58.04311170773853)
viewer.camera.perspective=0.0
#viewer.camera.interactive=True


In [ ]:
from skimage.measure import marching_cubes
import trimesh

# --- Parameters ---
num_cells, num_timepoints, Z, Y, X = croppings_array.shape

# Scaling factors
voxel_size_x = 0.691
voxel_size_y = 0.691
voxel_size_z = 2.0
scale = (voxel_size_z, voxel_size_y, voxel_size_x)
scale_matrix = np.diag([voxel_size_z, voxel_size_y, voxel_size_x, 1])

# --- Collect point clouds across all cells and timepoints ---
all_points = []
all_properties = {'cell_id': [], 'time': []}

for cell_idx in tqdm(range(num_cells)):
    for t in tqdm(range(num_timepoints)):
        mask = croppings_array[cell_idx, t]
        if np.any(mask):  # skip empty masks
            try:
                verts, faces, normals, values = marching_cubes(mask)
                mesh = trimesh.Trimesh(vertices=verts, faces=faces)
                # Apply scaling to the meshes before sampling
                mesh.apply_transform(scale_matrix)
                # Sample scaled meshes
                sampled_points = np.asarray(mesh.sample(1024))
                
            except Exception:
                sampled_points = np.empty((0, 3))
        else:
            sampled_points = np.empty((0, 3))
        
        # For each sampled point, prepend time and cell_idx
        if sampled_points.shape[0] > 0:
            time_and_cell = np.column_stack([
                np.full(sampled_points.shape[0], cell_idx),  # 'cell'
                np.full(sampled_points.shape[0], t),         # 'time'
                sampled_points                                # ZYX
            ])
            all_points.append(time_and_cell)

# Stack all point data
if all_points:
    points_array = np.vstack(all_points)
else:
    points_array = np.empty((0, 5))  # [cell, time, Z, Y, X]

# Convert properties to dict of arrays
all_properties = {k: np.array(v) for k, v in all_properties.items()}

viewer.add_labels(croppings_array, name='Segmentation', scale=scale)
viewer.add_points(points_array, name='Membrane clouds', size=0.75, face_color='white', ndim=5)

In [ ]:
viewer.camera.zoom=20
viewer.camera.angles=(-29.987043790992097, 49.57065942474976, -41.85955990611221)
viewer.camera.perspective=0.0
#viewer.camera.interactive=True

In [ ]:
mask = croppings_array

Set segmentations layer ('segmentation') as visible

### Figure 3B

In [ ]:
from PIL import Image

viewer.camera.zoom = 20
viewer.camera.angles = (-29.987043790992097, 49.57065942474976, -41.85955990611221)
viewer.camera.perspective = 0.0
# viewer.camera.interactive = True

# Loop over cell types (first dimension)
for c in range(0, 3):
    print(f'Processing celltype {celltype_dict[c]}')
    
    track_array = mask[c]  # shape: (T, Z, Y, X)

    if np.all(track_array == 0):
        continue

    last_valid_image = None
    for t in range(0, 360):  # timepoints
        subarray = track_array[t]  # shape: (Z, Y, X)

        if t == 0 or not np.all(subarray == 0):
            # Update viewer to display current timepoint
            viewer.dims.current_step = (c, t, 1, 74, 74)
            img = viewer.screenshot(scale=1, canvas_only=True, flash=False)
            image = Image.fromarray(img, mode="RGBA")
            last_valid_image = img  # store image array, not dims
        else:
            print(f'Subarray at (c={c}, t={t}) is all zeros, using last valid image.')
            if last_valid_image is not None:
                img = last_valid_image
                image = Image.fromarray(img, mode="RGBA")
            else:
                print("No valid image to fall back on yet.")
                continue

        # Save the image
        save_path = str((PATHS["anim_" + celltype_dict[c]]) / f"cell_t{t}.png")
        #image.save(save_path, format="PNG")


Set pointclouds layer ('membrane clouds') as visible

### Figure 3C

In [ ]:
from PIL import Image

viewer.camera.zoom = 20
viewer.camera.angles = (-29.987043790992097, 49.57065942474976, -41.85955990611221)
viewer.camera.perspective = 0.0
# viewer.camera.interactive = True

# Loop over cell types (first dimension)
for c in range(0, 3):
    print(f'Processing celltype {celltype_dict[c]}')
    
    track_array = mask[c]  # shape: (T, Z, Y, X)

    if np.all(track_array == 0):
        continue

    last_valid_image = None
    for t in range(0, 360):  # timepoints
        subarray = track_array[t]  # shape: (Z, Y, X)

        if t == 0 or not np.all(subarray == 0):
            # Update viewer to display current timepoint
            viewer.dims.current_step = (c, t, 1, 74, 74)
            img = viewer.screenshot(scale=1, canvas_only=True, flash=False)
            image = Image.fromarray(img, mode="RGBA")
            last_valid_image = img  # store image array, not dims
        else:
            print(f'Subarray at (c={c}, t={t}) is all zeros, using last valid image.')
            if last_valid_image is not None:
                img = last_valid_image
                image = Image.fromarray(img, mode="RGBA")
            else:
                print("No valid image to fall back on yet.")
                continue

        # Save the image
        save_path = str((PATHS["anim_" + celltype_dict[c] + "_pc"]) / f"cell_t{t}.png")
        image.save(save_path, format="PNG")


## Napari views & animation export (nucleus)

In [ ]:
celltype_dict = {0 : 'radial', 1 : 'basal', 2 : 'goblet'}

In [ ]:
%%capture
#import ncolor
nuc_mask = croppings_array_nuc
#mask_nc = ncolor.label(mask,max_depth=20)

import napari
viewer = napari.view_labels(nuc_mask)
voxel_size_x = 0.691 # um
voxel_size_y = 0.691 # um
voxel_size_z = 2 # um

reference_size = voxel_size_x

factor_z = voxel_size_z / reference_size
factor_y = voxel_size_y / reference_size
factor_x = voxel_size_x / reference_size

#viewer.layers['nuc_mask'].scale = [factor_z, factor_y, factor_x] # Z, Y, X order
viewer.dims.ndisplay = 3
#viewer.camera.center = [s//2 for s in mask.shape]
viewer.camera.zoom=5
viewer.camera.angles=(10.90517458968619, -20.777067798396835, 58.04311170773853)
viewer.camera.perspective=0.0
#viewer.camera.interactive=True


In [ ]:
from skimage.measure import marching_cubes
import trimesh


### Figure 3C

In [ ]:
# --- Parameters ---
num_cells, num_timepoints, Z, Y, X = croppings_array_nuc.shape

# Scaling factors
voxel_size_x = 0.691
voxel_size_y = 0.691
voxel_size_z = 2.0
scale = (voxel_size_z, voxel_size_y, voxel_size_x)
scale_matrix = np.diag([voxel_size_z, voxel_size_y, voxel_size_x, 1])

# --- Collect point clouds across all cells and timepoints ---
all_points = []
all_properties = {'cell_id': [], 'time': []}

for cell_idx in tqdm(range(num_cells)):
    for t in tqdm(range(num_timepoints)):
        mask = croppings_array_nuc[cell_idx, t]
        if np.any(mask):  # skip empty masks
            try:
                verts, faces, normals, values = marching_cubes(mask)
                mesh = trimesh.Trimesh(vertices=verts, faces=faces)
                # Apply scaling to the meshes before sampling
                mesh.apply_transform(scale_matrix)
                # Sample scaled meshes
                sampled_points = np.asarray(mesh.sample(1024))
                
            except Exception:
                sampled_points = np.empty((0, 3))
        else:
            sampled_points = np.empty((0, 3))
        
        # For each sampled point, prepend time and cell_idx
        if sampled_points.shape[0] > 0:
            time_and_cell = np.column_stack([
                np.full(sampled_points.shape[0], cell_idx),  # 'cell'
                np.full(sampled_points.shape[0], t),         # 'time'
                sampled_points                                # ZYX
            ])
            all_points.append(time_and_cell)

# Stack all point data
if all_points:
    points_array = np.vstack(all_points)
else:
    points_array = np.empty((0, 5))  # [cell, time, Z, Y, X]

# Convert properties to dict of arrays
all_properties = {k: np.array(v) for k, v in all_properties.items()}

viewer.add_labels(croppings_array_nuc, name='Segmentation', scale=scale)
viewer.add_points(points_array, name='Nuclei clouds', size=0.75, face_color='white', ndim=5)

### Figure 3B (nuclei)

In [ ]:
mask = croppings_array_nuc

In [ ]:
from PIL import Image

viewer.camera.zoom = 20
viewer.camera.angles = (-29.987043790992097, 49.57065942474976, -41.85955990611221)
viewer.camera.perspective = 0.0
# viewer.camera.interactive = True

# Loop over cell types (first dimension)
for c in range(0, 3):
    print(f'Processing celltype {celltype_dict[c]}')
    
    track_array = mask[c]  # shape: (T, Z, Y, X)

    if np.all(track_array == 0):
        continue

    last_valid_image = None
    for t in range(0, 360):  # timepoints
        subarray = track_array[t]  # shape: (Z, Y, X)

        if t == 0 or not np.all(subarray == 0):
            # Update viewer to display current timepoint
            viewer.dims.current_step = (c, t, 1, 74, 74)
            img = viewer.screenshot(scale=1, canvas_only=True, flash=False)
            image = Image.fromarray(img, mode="RGBA")
            last_valid_image = img  # store image array, not dims
        else:
            print(f'Subarray at (c={c}, t={t}) is all zeros, using last valid image.')
            if last_valid_image is not None:
                img = last_valid_image
                image = Image.fromarray(img, mode="RGBA")
            else:
                print("No valid image to fall back on yet.")
                continue

        # Save the image
        save_path = str((PATHS["anim_nuc_" + celltype_dict[c]]) / f"cell_t{t}.png")
        image.save(save_path, format="PNG")


### Figure 3C (Nuclei)

In [ ]:
viewer.camera.zoom = 20
viewer.camera.angles = (-29.987043790992097, 49.57065942474976, -41.85955990611221)
viewer.camera.perspective = 0.0
# viewer.camera.interactive = True

# Loop over cell types (first dimension)
for c in range(0, 3):
    print(f'Processing celltype {celltype_dict[c]}')
    
    track_array = mask[c]  # shape: (T, Z, Y, X)

    if np.all(track_array == 0):
        continue

    last_valid_image = None
    for t in range(0, 360):  # timepoints
        subarray = track_array[t]  # shape: (Z, Y, X)

        if t == 0 or not np.all(subarray == 0):
            # Update viewer to display current timepoint
            viewer.dims.current_step = (c, t, 1, 74, 74)
            img = viewer.screenshot(scale=1, canvas_only=True, flash=False)
            image = Image.fromarray(img, mode="RGBA")
            last_valid_image = img  # store image array, not dims
        else:
            print(f'Subarray at (c={c}, t={t}) is all zeros, using last valid image.')
            if last_valid_image is not None:
                img = last_valid_image
                image = Image.fromarray(img, mode="RGBA")
            else:
                print("No valid image to fall back on yet.")
                continue

        # Save the image
        save_path = str((PATHS["anim_nuc_" + celltype_dict[c] + "_pc"]) / f"cell_t{t}.png")
        image.save(save_path, format="PNG")


## Tracks overlay & animation export

### Figure 3A

In [55]:
import os

In [56]:
from tifffile import imread
import napari

# Load image
image = imread('D:/Mari_Sixth_Dataset_Analysis/nuclei_timelapses/timelapse_sixth_dataset.tif')
viewer = napari.view_image(image, name='nuclei', colormap='gray')

napari.run()


C:\Users\ghr283\AppData\Local\Temp\ipykernel_44116\2290571757.py:6: FutureWarning: `napari.view_image` is deprecated and will be removed in napari 0.7.0.
Use `viewer = napari.Viewer(); viewer.add_image(...)` instead.
  viewer = napari.view_image(image, name='nuclei', colormap='gray')


In [57]:
# Load and add tracks: radial
df = subset_df_nuc[subset_df_nuc['Spot track ID relabelled'] == 63].rename(columns={
    'Track ID_y': 'track_id', 't': 'frame',
    'Z_orig': 'z', 'Y_orig': 'y', 'X_orig': 'x'
}).sort_values(by=['track_id', 'frame'])

tracks_data = df[['track_id', 'frame', 'z', 'y', 'x']].to_numpy()
# Make a constant-valued property array (e.g., all ones)
n_rows = tracks_data.shape[0]
ones = np.ones(n_rows)  # or any value or pattern
ones[0] = 0.0000001

properties = {
    'ones': ones
}

viewer.add_tracks(tracks_data, properties = properties, name='radial tracks')

# Goblet
df = subset_df_nuc[subset_df_nuc['Spot track ID relabelled'] == 2690].rename(columns={
    'Track ID_y': 'track_id', 't': 'frame',
    'Z_orig': 'z', 'Y_orig': 'y', 'X_orig': 'x'
}).sort_values(by=['track_id', 'frame'])
# Reassign track IDs to be consecutive integers per group

tracks_data = df[['track_id', 'frame', 'z', 'y', 'x']].to_numpy()

# Make a constant-valued property array (e.g., all ones)
n_rows = tracks_data.shape[0]
ones = np.ones(n_rows)  # or any value or pattern
ones[0] = 0.0000001

properties = {
    'ones': ones
}

viewer.add_tracks(tracks_data, properties=properties, name='goblet tracks')

# Basal
df = subset_df_nuc[subset_df_nuc['Spot track ID relabelled'] == 225].rename(columns={
    'Track ID_y': 'track_id', 't': 'frame',
    'Z_orig': 'z', 'Y_orig': 'y', 'X_orig': 'x'
}).sort_values(by=['track_id', 'frame'])
# Reassign track IDs to be consecutive integers per group

tracks_data = df[['track_id', 'frame', 'z', 'y', 'x']].to_numpy()

# Make a constant-valued property array (e.g., all ones)
n_rows = tracks_data.shape[0]
ones = np.ones(n_rows)  # or any value or pattern
ones[0] = 0.0000001

properties = {
    'ones': ones
}

viewer.add_tracks(tracks_data, properties=properties, name='basal tracks')

<Tracks layer 'basal tracks' at 0x2239db47880>

In [211]:
for t in range(0, 360):
    # Update viewer to display the current subarray
    viewer.dims.current_step = (t, 8, 1071, 1071)
    img = viewer.screenshot(scale=1, canvas_only=True, flash=False)
    # Convert the NumPy array to a Pillow Image
    image = Image.fromarray(img, mode="RGBA")
    
    # Save the image as a PNG
    save_path = str(PATHS["tracks_anim_dir"] / f"t{t}.png")
    image.save(save_path, format="PNG")